# Phase 20 — interpretable multiscale regime baseline

This notebook defines the deterministic causal comparator that every probabilistic model must beat in usefulness, not merely rename. It separates **structural shape** from **direction** and does not use future outcomes or returns to assign a regime.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from dc_multiscale_research import (
    FeatureTier,
    ObservationClock,
    REGIME_ORDER,
    build_multiscale_state,
    dwell_lengths,
    load_close,
    model_frame,
    plot_price_states,
    ratio_ladder,
    transition_matrix,
)

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
FIRING_WINDOW_BARS = 240
LAST_N = 12_000

## Frozen direct taxonomy

The primary label has four direction-neutral structural classes:

- `broad_coherence`: no adjacent disagreement;
- `shallow_counter`: one boundary and at most half the scales have joined the small-scale counter-move;
- `deep_counter`: one boundary and more than half the scales have joined it;
- `fragmented`: two or more boundaries, so one propagation-depth number is invalid.

`outer_mode`, `small_mode`, and `directed_regime` retain direction separately. The 50% boundary is specified before any outcome analysis and generalizes to arbitrary $N$.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
catalog, global_state, scale_state = build_multiscale_state(
    prices, THRESHOLDS, firing_window=FIRING_WINDOW_BARS, include_scale_state=True
)
print(f"{SYMBOL}: {len(prices):,} prices; {len(global_state):,} initialized hierarchy bars")
print(f"Range: {global_state.index[0]} to {global_state.index[-1]}")
display(catalog.style.format({"threshold": "{:.4%}", "ratio_to_lower": "{:.3f}"}))

GBPUSD.r: 258,573 prices; 257,813 initialized hierarchy bars
Range: 2023-05-25 05:20:00+02:00 to 2026-11-09 13:30:00+02:00


,threshold_index,threshold_id,threshold,ratio_to_lower
threshold_index,,,,
0,0,T01,0.2000%,nan
1,1,T02,0.3000%,1.500
2,2,T03,0.4500%,1.500
3,3,T04,0.6750%,1.500
4,4,T05,1.0125%,1.500


## Baseline occupancy under both observation clocks

Bar time measures persistence. Union-event time measures the population at bars where at least one threshold changed mode. Neither is treated as the hidden truth.

In [3]:
bar_occupancy = global_state["structural_regime"].value_counts(normalize=True).reindex(REGIME_ORDER, fill_value=0.0)
event_state = global_state.loc[global_state["union_event"]]
event_occupancy = event_state["structural_regime"].value_counts(normalize=True).reindex(REGIME_ORDER, fill_value=0.0)
occupancy = pd.concat({
    ObservationClock.BAR.value: bar_occupancy,
    ObservationClock.UNION_EVENT.value: event_occupancy,
}, axis=1)
directed_occupancy = global_state["directed_regime"].value_counts(normalize=True).rename("bar_share")
display(occupancy.style.format("{:.2%}"))
display(directed_occupancy.to_frame().style.format("{:.2%}"))

,bar_time,union_event_time
structural_regime,,
broad_coherence,26.39%,15.76%
shallow_counter,22.56%,27.02%
deep_counter,21.68%,18.93%
fragmented,29.37%,38.29%


,bar_share
directed_regime,
fragmented_outer_up,16.61%
broad_up,14.90%
fragmented_outer_down,12.76%
shallow_counter_down_against_up,11.90%
broad_down,11.49%
deep_counter_up_against_down,11.18%
shallow_counter_up_against_down,10.66%
deep_counter_down_against_up,10.50%


## What each direct regime contains now

These are contemporaneous causal measurements. The table deliberately contains no forward return, later propagation, rejection, or terminal-depth field.

In [4]:
current_columns = [
    "fragmentation",
    "counter_depth_fraction",
    "mean_mode_age_bars",
    "mean_reversal_progress",
    "max_reversal_progress",
    "hierarchy_dcc_rate",
]
current_profile = global_state.groupby("structural_regime", observed=True)[current_columns].agg(
    ["count", "median", "mean"]
).reindex(REGIME_ORDER)
display(current_profile)

direction_balance = pd.crosstab(
    global_state["structural_regime"], global_state["outer_mode"], normalize="index"
).reindex(REGIME_ORDER)
display(direction_balance.style.format("{:.2%}"))

fragmentation                  counter_depth_fraction  \
                          count median      mean                  count   
structural_regime                                                         
broad_coherence           68033   0.00  0.000000                  68033   
shallow_counter           58168   0.25  0.250000                  58168   
deep_counter              55890   0.25  0.250000                  55890   
fragmented                75722   0.50  0.555757                      0   

                                   mean_mode_age_bars                     \
                  median      mean              count median        mean   
structural_regime                                                          
broad_coherence      0.0  0.000000              68033  486.8  626.980136   
shallow_counter      0.2  0.299006              58168  515.5  670.582159   
deep_counter         0.6  0.693612              55890  460.2  600.545647   
fragmented           NaN       NaN              75722  475.2  612.164755   

                  mean_reversal_progress                      \
                                   count    median      mean   
structural_regime                                              
broad_coherence                    68033  0.161534  0.180549   
shallow_counter                    58168  0.378475  0.379570   
deep_counter                       55890  0.345609  0.350555   
fragmented                         75722  0.424770  0.414449   

                  max_reversal_progress                      \
                                  count    median      mean   
structural_regime                                             
broad_coherence                   68033  0.296981  0.334568   
shallow_counter                   58168  0.699453  0.704249   
deep_counter                      55890  0.743131  0.749130   
fragmented                        75722  0.723401  0.730630   

                  hierarchy_dcc_rate                      
                               count    median      mean  
structural_regime                                         
broad_coherence                68033  0.003333  0.003665  
shallow_counter                58168  0.003333  0.003773  
deep_counter                   55890  0.003333  0.004178  
fragmented                     75722  0.004167  0.004483

outer_mode,down,up
structural_regime,,
broad_coherence,43.55%,56.45%
shallow_counter,47.25%,52.75%
deep_counter,51.58%,48.42%
fragmented,43.45%,56.55%


## Persistence and transitions are descriptions, not model assumptions

In [5]:
baseline_codes = global_state["structural_regime_code"].to_numpy(dtype=np.int64)
baseline_transition = pd.DataFrame(
    transition_matrix(baseline_codes, len(REGIME_ORDER)), index=REGIME_ORDER, columns=REGIME_ORDER
)
dwell_records = []
for regime_code, regime_name in enumerate(REGIME_ORDER):
    runs = []
    states = baseline_codes
    starts = np.flatnonzero(np.r_[True, states[1:] != states[:-1]])
    ends = np.r_[starts[1:], len(states)]
    for start, end in zip(starts, ends, strict=True):
        if states[start] == regime_code:
            runs.append(end - start)
    dwell_records.append({
        "structural_regime": regime_name,
        "runs": len(runs),
        "median_dwell_bars": np.median(runs) if runs else np.nan,
        "p90_dwell_bars": np.quantile(runs, 0.90) if runs else np.nan,
    })
display(baseline_transition.style.format("{:.3f}"))
display(pd.DataFrame.from_records(dwell_records).set_index("structural_regime"))

,broad_coherence,shallow_counter,deep_counter,fragmented
broad_coherence,0.988,0.012,0.000,0.000
shallow_counter,0.005,0.985,0.006,0.005
deep_counter,0.003,0.000,0.987,0.011
fragmented,0.005,0.001,0.005,0.989


,runs,median_dwell_bars,p90_dwell_bars
structural_regime,,,
broad_coherence,798,51.0,207.0
shallow_counter,872,43.0,170.7
deep_counter,748,44.0,193.3
fragmented,868,44.5,223.6


## Feature tiers available to later models

The baseline label is not supplied as a model feature. The tiers make later ablation explicit: structural geometry first, live dynamics second, and previous completed-leg measurements last.

In [6]:
tier_records = []
for tier in FeatureTier:
    _, columns = model_frame(global_state, tier)
    tier_records.append({"tier": tier.value, "features": len(columns), "columns": columns})
display(pd.DataFrame.from_records(tier_records).set_index("tier"))

for forbidden in ("future", "terminal", "outcome", "resolution"):
    if any(forbidden in column for record in tier_records for column in record["columns"]):
        raise AssertionError(f"Forbidden model feature token found: {forbidden}")
print("Model-feature hindsight-name audit: passed")

,features,columns
tier,,
structural,20,"[mode_sign_00, reversal_progress_00, log1p_mod..."
dynamic,36,"[mode_sign_00, reversal_progress_00, log1p_mod..."
full,51,"[mode_sign_00, reversal_progress_00, log1p_mod..."


Model-feature hindsight-name audit: passed


In [7]:
plot_price_states(
    global_state,
    "directed_regime",
    f"{SYMBOL}: direct causal multiscale regime baseline",
    last_n=LAST_N,
).show()

# Phase 20 completion gate

Before probabilistic modelling, be able to explain:

1. Why structural shape and direction are separate axes.
2. Why broad coherence is not automatically a profitable trend.
3. Why shallow and deep counter-moves are defined by a scale fraction rather than future propagation.
4. Why fragmented structure has no valid single propagation depth.
5. Why bar-time and event-time occupancy differ.
6. Why the direct taxonomy is a comparator, not ground truth.
7. Why later models must add probabilistic confidence, persistence, compression, or useful separation rather than merely reproduce these names.